# LLM Fine-Tune Platform — Train + Eval (Colab T4)

**Before running:** Runtime -> Change runtime type -> **T4 GPU**.

**Prereqs:**
1. Request access to `meta-llama/Llama-3.2-3B-Instruct` on Hugging Face (gated).
2. Add secrets in Colab (left sidebar -> key icon): `HF_TOKEN` (required), `WANDB_API_KEY` (optional), `ANTHROPIC_API_KEY` (optional, for LLM-judge).

Then: Runtime -> **Run all**.

In [ ]:
# 1. GPU check — must show a Tesla T4 (or similar)
!nvidia-smi

In [ ]:
# 2. Clone repo + install deps
#    requirements-colab.txt keeps Colab's CUDA torch and adds only the
#    fine-tune libs. The old version caps drag torch down to 2.5.1, which
#    mismatches Colab's prebuilt torchvision 0.26 (compiled for torch 2.11)
#    -> "operator torchvision::nms does not exist" at transformers import.
#    We don't use torchvision (text-only), so remove it to skip that import.
!git clone https://github.com/PearlKalariya/LLM_FIneTunning.git
%cd LLM_FIneTunning
!pip install -q -r requirements-colab.txt
!pip uninstall -y -q torchvision

In [ ]:
# 3. Load secrets from Colab's Secrets panel into env
import os
from google.colab import userdata

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
for opt in ('WANDB_API_KEY', 'ANTHROPIC_API_KEY'):
    try:
        os.environ[opt] = userdata.get(opt)
    except Exception:
        print(f'[skip] {opt} not set — continuing without it')

# no W&B key -> run offline instead of prompting
if not os.environ.get('WANDB_API_KEY'):
    os.environ['WANDB_MODE'] = 'offline'

!huggingface-cli login --token $HF_TOKEN

In [ ]:
# 4. Build dataset -> data/processed/{train,val,test}.jsonl
#    Expect: train 126 / val 15 / test 18
!python -m src.data_prep

In [ ]:
# 5. QLoRA fine-tune (~15-40 min on T4). Watch the loss print.
#    Adapter saved to adapters/financial_extraction/ + registry entry.
!python -m src.train --task financial_extraction

In [ ]:
# 6. Eval baseline vs fine-tuned -> prints table, writes eval_results.json
!python -m src.evaluate --task financial_extraction

In [ ]:
# 7. Show results — copy these numbers into docs/EVALUATION.md
import json
print(json.dumps(json.load(open('eval_results.json')), indent=2))

In [ ]:
# 8. (optional) Download the trained adapter to your machine
!zip -r adapter.zip adapters/financial_extraction
from google.colab import files
files.download('adapter.zip')